# Pokémon: un solo modello XGBoost

**Architettura:** primi 30 turni → riepilogo dello stato e informazioni iniziali → XGBoost → probabilità di vittoria del giocatore 1.

Usiamo **300 alberi, profondità massima 3 e learning rate 0,05**. I parametri sono fissi. Ogni battaglia diventa una riga con Pokémon osservati ed esausti, ultimi HP osservati e loro anzianità, stato dei Pokémon attivi e caratteristiche iniziali disponibili.

Il notebook è pronto da eseguire: **avvii tu l'addestramento nella sezione 5**. La sezione 7 riaddestra lo stesso modello su tutto il training per generare **un solo file finale: `submission.csv`**. Le funzioni sono spiegate nel modulo `pokemon_competition.py`.

## 1. Preparare il progetto

In locale scegli l'ambiente Python `.venv` già presente nella cartella FDS. Questa cella importa le funzioni senza addestrare il modello.

Su Kaggle abilita Internet, collega i dati della competizione e inserisci `REPOSITORY_URL` e `REPOSITORY_REVISION` qui sotto. Il notebook clona il repository del team e carica il modulo dal commit indicato.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = os.environ.get("POKEMON_REPOSITORY_URL", "")
REPOSITORY_REVISION = os.environ.get("POKEMON_REPOSITORY_REVISION", "")
RUNNING_ON_KAGGLE = Path("/kaggle/working").is_dir()

if RUNNING_ON_KAGGLE:
    if not REPOSITORY_URL or not REPOSITORY_REVISION:
        raise ValueError("Imposta REPOSITORY_URL e REPOSITORY_REVISION prima di eseguire su Kaggle.")
    repository_directory = Path("/kaggle/working/pokemon_repository")
    if not repository_directory.exists():
        subprocess.run(["git", "clone", REPOSITORY_URL, str(repository_directory)], check=True)
    else:
        existing_url = subprocess.check_output(
            ["git", "-C", str(repository_directory), "remote", "get-url", "origin"], text=True
        ).strip()
        if existing_url != REPOSITORY_URL:
            raise ValueError("La cartella esistente appartiene a un repository diverso.")
        subprocess.run(["git", "-C", str(repository_directory), "fetch", "origin"], check=True)
    subprocess.run(
        ["git", "-C", str(repository_directory), "checkout", "--detach", REPOSITORY_REVISION], check=True
    )
    module_directory = repository_directory
    if not (module_directory / "pokemon_competition.py").is_file():
        module_directory = repository_directory / "soluzione"
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "-r", str(module_directory / "requirements.txt")],
        check=True,
    )
else:
    module_directory = Path.cwd()
    if not (module_directory / "pokemon_competition.py").is_file():
        module_directory = module_directory / "soluzione"

if not (module_directory / "pokemon_competition.py").is_file():
    raise FileNotFoundError("Apri il notebook dalla cartella soluzione o dalla cartella FDS.")
sys.path.insert(0, str(module_directory))

import json
import pandas
import pokemon_competition as competition

print("Modulo:", module_directory / "pokemon_competition.py")
print(json.dumps(competition.package_versions(), indent=2))

## 2. Scegliere i percorsi

`DATA_DIRECTORY` indica dove sono i dati. `OUTPUT_DIRECTORY` indica dove saranno salvati i tuoi risultati. Puoi modificare queste variabili. Il modello usa la CPU.

In [ ]:
if RUNNING_ON_KAGGLE:
    required_files = {"train.jsonl", "test.jsonl", "sample_submission.csv"}
    candidate_directories = []
    for path in Path("/kaggle/input").rglob("train.jsonl"):
        available_files = {child.name for child in path.parent.iterdir() if child.is_file()}
        if required_files.issubset(available_files):
            candidate_directories.append(path.parent)
    if len(candidate_directories) != 1:
        raise ValueError("Imposta DATA_DIRECTORY: non è stata trovata una sola cartella dati.")
    DATA_DIRECTORY = candidate_directories[0]
    OUTPUT_DIRECTORY = Path("/kaggle/working/results")
else:
    DATA_DIRECTORY = module_directory.parent / "challenge" / "dataset"
    OUTPUT_DIRECTORY = module_directory / "results"

DATA_DIRECTORY = Path(os.environ.get("POKEMON_DATA_DIRECTORY", str(DATA_DIRECTORY)))
OUTPUT_DIRECTORY = Path(os.environ.get("POKEMON_OUTPUT_DIRECTORY", str(OUTPUT_DIRECTORY)))
NUMBER_OF_THREADS = 4
print("Dati:", DATA_DIRECTORY)
print("Risultati:", OUTPUT_DIRECTORY)

## 3. Leggere e controllare il training

Leggiamo le battaglie con il vincitore conosciuto e rimuoviamo in memoria le copie esatte prima di dividere i dati. Restano 9.986 battaglie uniche. Le etichette fornite vengono conservate, compresa la possibile anomalia della battaglia 5718. I file originali restano invariati.

In [ ]:
raw_training_battles = competition.load_battles(DATA_DIRECTORY / "train.jsonl")
training_battles, removed_rows = competition.remove_duplicate_battles(raw_training_battles)
del raw_training_battles

print("Battaglie uniche:", len(training_battles))
print("Copie rimosse:", len(removed_rows))
labels = pandas.Series([battle["player_won"] for battle in training_battles])
print("Distribuzione delle etichette:")
print(labels.value_counts().sort_index())

## 4. Capire le caratteristiche

Ricostruiamo l'ultimo stato osservato di ogni Pokémon. Contiamo una sola volta gli esausti. Gli HP sconosciuti restano mancanti; XGBoost li gestisce direttamente. Ogni misura di HP è accompagnata dal tempo trascorso dalla sua osservazione.

Aggiungiamo stato e modifiche alle statistiche dei Pokémon attivi, statistiche aggregate della squadra iniziale del giocatore 1, caratteristiche disponibili del Pokémon iniziale avversario e differenze tra quantità confrontabili. Le categorie vengono trasformate in colonne binarie dalla codifica del modello.

La somma degli HP riguarda soltanto i membri osservati. L'identificativo e l'etichetta non sono caratteristiche. La cella seguente mostra un esempio e **non avvia il training**.

In [ ]:
example_features = competition.extract_features(training_battles[0])
example_names = [
    "p1_observed_pokemon", "p2_observed_pokemon",
    "p1_fainted_pokemon", "p2_fainted_pokemon",
    "p1_observed_health_sum", "p2_observed_health_sum",
    "p1_health_observation_age_max", "p2_health_observation_age_max",
    "p1_active_health", "p2_active_health",
]
display(pandas.DataFrame({
    "caratteristica": example_names,
    "valore": [example_features[name] for name in example_names],
}))
print("Parametri fissi:", competition.MODEL_PARAMETERS)

## 5. Addestrare e valutare XGBoost

**Il primo addestramento parte quando esegui la cella seguente.**

Usiamo l'80% delle battaglie uniche per addestrare XGBoost e il 20% per misurare quante previsioni sono corrette. La divisione conserva circa le proporzioni delle due classi ed è ripetibile grazie al seed 42. La codifica impara le categorie soltanto dalla parte di addestramento.

Non confrontiamo altri modelli o configurazioni. La soglia resta 0,5. La matrice di confusione ha le classi reali sulle righe e le classi predette sulle colonne. Il risultato e gli identificativi dei due sottoinsiemi vengono salvati in `validation.json`. Questo risultato serve a valutare l'architettura fissata, non a modificare i parametri sulla stessa validazione.

In [ ]:
validation_result = competition.validate_model(
    training_battles, OUTPUT_DIRECTORY, NUMBER_OF_THREADS
)
print("Battaglie per il training:", validation_result["training_rows"])
print("Battaglie per la validazione:", validation_result["validation_rows"])
print(f"Accuracy di validazione: {validation_result['accuracy']:.2%}")
display(pandas.DataFrame(
    validation_result["confusion_matrix"],
    index=["Reale 0", "Reale 1"],
    columns=["Predetto 0", "Predetto 1"],
))

## 6. Caricare il test

Il test contiene battaglie senza vincitore conosciuto. Carichiamo anche il file di esempio, che stabilisce gli identificativi e l'ordine delle righe da consegnare.

In [ ]:
test_battles = competition.load_battles(DATA_DIRECTORY / "test.jsonl", has_labels=False)
sample_submission = pandas.read_csv(DATA_DIRECTORY / "sample_submission.csv")
print("Battaglie di test:", len(test_battles))

## 7. Addestrare il modello finale e creare la submission

**Questa cella avvia l'addestramento finale.** Usiamo lo stesso XGBoost con gli stessi parametri, questa volta su tutte le battaglie uniche del training. È la stessa architettura valutata al passaggio 5; ora usiamo tutti gli esempi etichettati per prevedere il test.

Una probabilità almeno pari a 0,5 diventa `1`, altrimenti `0`. La funzione controlla le etichette e l'ordine degli identificativi e salva:

- `submission.csv`: l'unico CSV finale, da caricare nella competizione;
- `model.joblib`: il modello addestrato, compresa la codifica;
- `model_information.json`: parametri, seed, versioni e informazioni per riprodurre il risultato.

I file vengono salvati nella cartella scelta e non vengono inviati automaticamente a Kaggle.

In [ ]:
submission = competition.train_and_create_submission(
    training_battles, test_battles, sample_submission,
    OUTPUT_DIRECTORY, NUMBER_OF_THREADS,
)
display(submission.head())
print("File finale:", OUTPUT_DIRECTORY / "submission.csv")
print("Numero di previsioni:", len(submission))
print(submission["player_won"].value_counts().sort_index())